# Phase 2 L-NER — Knowledge Distillation (BIO + CRF)

**Teacher**: Best NER model (decoder-based, 150M params)
**Student**: Encoder-based NER model (~48M params)

Distills teacher's soft emission logits into a smaller student model.

**Colab + Drive version. Needs: trained teacher .pt, best_encdec.pt, bpe_40k.json, a1_tokenizer.py in Drive.**

In [ ]:
import os, sys, re, json, math, time, random, bisect, copy, zipfile, glob, shutil
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

print("Imports OK")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1 — Configs (Teacher = Decoder, Student = Encoder)

In [ ]:
# ── Teacher config (decoder-only, 150M) ──
@dataclass
class TeacherConfig:
    vocab_size: int = 40_000
    d_model: int = 768
    n_layers: int = 19
    n_query_heads: int = 12
    n_kv_heads: int = 4
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    weight_tying: bool = True
    rope_theta: float = 10000.0
    bio_num_tags: int = 25

    def __post_init__(self):
        self.head_dim = self.d_model // self.n_query_heads
        self.n_groups = self.n_query_heads // self.n_kv_heads

# ── Student config (encoder, ~48M) ──
@dataclass
class StudentConfig:
    vocab_size: int = 40_000
    d_model: int = 640
    n_layers: int = 5
    n_heads: int = 10
    n_kv_heads: int = 5
    d_ff: int = 1728
    max_seq_len: int = 512
    rms_eps: float = 1e-5
    rope_theta: float = 10000.0
    dropout: float = 0.1
    bio_num_tags: int = 25

    def __post_init__(self):
        self.head_dim = self.d_model // self.n_heads

# ── NER / Training config ──
@dataclass
class NERConfig:
    num_labels: int = 12
    bio_num_tags: int = 25
    window_size: int = 510
    window_stride: int = 255
    lr: float = 1e-5
    min_lr: float = 1e-7
    warmup_steps: int = 200
    num_epochs: int = 25
    batch_size: int = 4
    grad_accum: int = 4
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_amp: bool = True
    seed: int = 42
    # KD hyperparams
    kd_alpha: float = 0.5       # weight for hard CRF loss
    kd_temperature: float = 2.0 # temperature for soft targets

CFG = NERConfig()
T_CFG = TeacherConfig()
S_CFG = StudentConfig()

OUT = Path(".")
CKPT_DIR = OUT / "ner_checkpoints"; CKPT_DIR.mkdir(exist_ok=True)
SUB_DIR = OUT / "submissions"; SUB_DIR.mkdir(exist_ok=True)
print(f"KD: alpha={CFG.kd_alpha} (hard), temp={CFG.kd_temperature}")

KD: alpha=0.5 (hard), temp=2.0


In [ ]:
# ── Search Drive for required files ──
def find_in_drive(filename):
    """Search Google Drive for a file by name."""
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if filename in files:
            return os.path.join(root, filename)
    return None

# Teacher checkpoint — try several names
TEACHER_CKPT = None
for tname in ["exp1_full_ft_bio_fixed_best.pt", "teacher_full_ft.pt",
              "exp1_full_ft_flat_fixed_best.pt", "exp_encoder_bio_best.pt"]:
    TEACHER_CKPT = find_in_drive(tname)
    if TEACHER_CKPT:
        break

ENCDEC_CKPT   = find_in_drive("bestencdec.pt")
tok_json_path = find_in_drive("bpe_40k.json")
tok_py_path   = find_in_drive("a1_tokenizer.py")

for f, desc in [(TEACHER_CKPT, "Teacher NER ckpt"), (ENCDEC_CKPT, "best_encdec.pt"),
                (tok_json_path, "bpe_40k.json"), (tok_py_path, "a1_tokenizer.py")]:
    assert f is not None and os.path.isfile(f), f"Not found in Drive: {desc}"
    print(f"  Found {desc}: {f}")

sys.path.insert(0, os.path.dirname(tok_py_path))
from a1_tokenizer import BPETokenizer
tokenizer = BPETokenizer.load(tok_json_path)
T_CFG.vocab_size = S_CFG.vocab_size = tokenizer.vocab_size
print(f"Tokenizer loaded: vocab={tokenizer.vocab_size}")

  Found Teacher NER ckpt: /content/drive/MyDrive/teacher_full_ft.pt
  Found best_encdec.pt: /content/drive/MyDrive/bestencdec.pt
  Found bpe_40k.json: /content/drive/MyDrive/bpe_40k.json
  Found a1_tokenizer.py: /content/drive/MyDrive/a1_tokenizer.py
Tokenizer loaded: vocab=40000


In [ ]:
!pip -q install huggingface_hub
from huggingface_hub import hf_hub_download

HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = ""

NER_FILES = {
    "fold1_text": "Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text": "Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold": "Phase_2_LNER/Train/fold2-gold.json",
    "labels": "Phase_2_LNER/Train/labels.txt",
    "fold3_text": "Phase_2_LNER/Test/fold3-text.json",
}

ner_paths = {}
for key, fn in NER_FILES.items():
    ner_paths[key] = hf_hub_download(HF_REPO, fn, repo_type="dataset", token=HF_TOKEN)
    print(f"  {key:12s} -> downloaded")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

  fold1_text   -> downloaded


fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

  fold1_gold   -> downloaded


fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

  fold2_text   -> downloaded


... (2 outputs left out)

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

  labels       -> downloaded


fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

  fold3_text   -> downloaded


In [ ]:
# Labels + BIO tags
with open(ner_paths["labels"], "r") as f:
    LABELS = [l.strip() for l in f if l.strip()]

O_TAG = 0
tag2id = {"O": O_TAG}
id2tag = {O_TAG: "O"}
for i, lab in enumerate(LABELS):
    tag2id[f"B-{lab}"] = 2*i + 1
    tag2id[f"I-{lab}"] = 2*i + 2
    id2tag[2*i + 1] = f"B-{lab}"
    id2tag[2*i + 2] = f"I-{lab}"
print(f"BIO tags: {len(tag2id)}")

def load_json(p):
    with open(p) as f:
        return json.load(f)

fold1_text = load_json(ner_paths["fold1_text"])
fold1_gold = load_json(ner_paths["fold1_gold"])
fold2_text = load_json(ner_paths["fold2_text"])
fold2_gold = load_json(ner_paths["fold2_gold"])
fold3_text = load_json(ner_paths["fold3_text"])

BIO tags: 25


## 2 — Shared Utilities (offset encoder, BIO tags, dataset, eval)

In [ ]:
def encode_with_offsets(tokenizer, text):
    token_ids = []
    char_spans = []
    for m in re.finditer(r'\S+', text):
        word = m.group()
        ws = m.start()
        char_byte_bounds = [0]
        for ch in word:
            char_byte_bounds.append(char_byte_bounds[-1] + len(ch.encode('utf-8')))
        bpe_syms = tokenizer._bpe(word)
        SPACE_BYTES = 1
        fb_pos = 0
        for sym in bpe_syms:
            tok_id = tokenizer.token_to_id.get(sym, tokenizer.token_to_id.get('<unk>', 1))
            n_bytes = len(sym)
            fb_end = fb_pos + n_bytes
            wb_s = max(0, fb_pos - SPACE_BYTES)
            wb_e = max(0, fb_end - SPACE_BYTES)
            wb_e = min(wb_e, char_byte_bounds[-1])
            wb_s = min(wb_s, char_byte_bounds[-1])
            if wb_e > wb_s:
                c_s = bisect.bisect_right(char_byte_bounds, wb_s) - 1
                c_e = bisect.bisect_left(char_byte_bounds, wb_e)
                char_spans.append((ws + max(0, c_s), ws + min(len(word), c_e)))
            else:
                char_spans.append((ws, ws))
            token_ids.append(tok_id)
            fb_pos = fb_end
    return token_ids, char_spans

def get_doc_text(d):
    if isinstance(d, str):
        return d
    if isinstance(d, dict):
        return d.get("text", "")
    return str(d)

def assign_bio_tags(char_spans, gold_annotations, tag2id):
    tags = [O_TAG] * len(char_spans)
    for ann in sorted(gold_annotations, key=lambda a: int(a[0])):
        gs, ge, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in LABELS:
            continue
        b_tag = tag2id.get(f"B-{label}", O_TAG)
        i_tag = tag2id.get(f"I-{label}", O_TAG)
        first = True
        for t_idx, (cs, ce) in enumerate(char_spans):
            if ce <= cs:
                continue
            if cs < ge and ce > gs:
                tags[t_idx] = b_tag if first else i_tag
                first = False
    return tags


class NERDataset(Dataset):
    def __init__(self, text_dict, gold_dict, tokenizer, cfg, tag2id):
        self.windows = []
        self.cfg = cfg
        ws = cfg.window_size
        stride = cfg.window_stride if cfg.window_stride > 0 else ws
        for doc_id in tqdm(text_dict, desc="Building dataset"):
            text = get_doc_text(text_dict[doc_id])
            if not text.strip():
                continue
            all_ids, all_spans = encode_with_offsets(tokenizer, text)
            n = len(all_ids)
            if n == 0:
                continue
            all_tags = assign_bio_tags(all_spans, gold_dict.get(doc_id, []), tag2id)
            starts = list(range(0, max(1, n - ws + 1), stride))
            if starts[-1] + ws < n:
                starts.append(n - ws)
            for wi, s in enumerate(starts):
                e = min(s + ws, n)
                self.windows.append((all_ids[s:e], all_tags[s:e], doc_id, wi))
        print(f"  {len(self.windows)} windows")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w_ids, w_tags, _, _ = self.windows[idx]
        L = self.cfg.window_size
        pad = L - len(w_ids)
        return {
            "input_ids": torch.tensor(w_ids + [0]*pad, dtype=torch.long),
            "labels": torch.tensor(w_tags + [-100]*pad, dtype=torch.long),
            "attention_mask": torch.tensor([1]*len(w_ids) + [0]*pad, dtype=torch.long),
        }

print("Utilities defined")

Utilities defined


## 3 — Teacher Model (Decoder + BiLSTM + CRF)

In [ ]:
# ── Decoder model architecture (must match Phase 1 decoder) ──
class RMSNorm_Dec(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return (x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)).to(x.dtype) * self.weight

def precompute_rope_dec(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()

def apply_rope_dec(x, cos, sin, offset=0):
    T = x.size(2)
    c = cos[offset:offset+T].unsqueeze(0).unsqueeze(0)
    s = sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rotated = torch.cat([-x2, x1], dim=-1)
    c_exp = torch.cat([c.expand(-1, -1, -1, hd), c.expand(-1, -1, -1, hd)], dim=-1)
    s_exp = torch.cat([s.expand(-1, -1, -1, hd), s.expand(-1, -1, -1, hd)], dim=-1)
    return x * c_exp + rotated * s_exp

class GQA_Dec(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nqh = cfg.n_query_heads
        self.nkh = cfg.n_kv_heads
        self.ng = cfg.n_groups
        self.hd = cfg.head_dim
        self.dm = cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.n_query_heads * cfg.head_dim, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.o = nn.Linear(cfg.n_query_heads * cfg.head_dim, cfg.d_model, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)

    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nqh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nkh, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nkh, self.hd).transpose(1, 2)
        q = apply_rope_dec(q, cos, sin, offset)
        k = apply_rope_dec(k, cos, sin, offset)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        nc = (k, v)
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B, self.nkh, self.ng, k.size(2), self.hd).reshape(B, self.nqh, -1, self.hd)
            v = v.unsqueeze(2).expand(B, self.nkh, self.ng, v.size(2), self.hd).reshape(B, self.nqh, -1, self.hd)
        w = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if mask is not None:
            w = w + mask
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        return self.resid_drop(self.o(torch.matmul(w, v).transpose(1, 2).contiguous().view(B, T, self.dm))), nc

class SwiGLU_Dec(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class Block_Dec(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm_Dec(cfg.d_model)
        self.attn = GQA_Dec(cfg)
        self.norm2 = RMSNorm_Dec(cfg.d_model)
        self.ffn = SwiGLU_Dec(cfg)

    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        h, cache = self.attn(self.norm1(x), cos, sin, mask, kv_cache, offset)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x, cache

class LegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([Block_Dec(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm_Dec(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.weight_tying:
            self.lm_head.weight = self.tok_emb.weight
        cos, sin = precompute_rope_dec(cfg.head_dim, cfg.max_seq_len * 2, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _causal_mask(self, T, device, S=None):
        if S is None:
            S = T
        m = torch.full((T, S), float("-inf"), device=device)
        for i in range(T):
            m[i, :S - T + i + 1] = 0.0
        return m.unsqueeze(0).unsqueeze(0)


# ── Teacher NER model (Decoder + BiLSTM + CRF) ──
class CRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags))
        self.start_transitions = nn.Parameter(torch.randn(num_tags))
        self.end_transitions = nn.Parameter(torch.randn(num_tags))
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)

    def forward(self, emissions, tags, mask):
        return (self._forward_alg(emissions, mask) - self._score_sentence(emissions, tags, mask)).mean()

    def _score_sentence(self, emissions, tags, mask):
        B, T, C = emissions.shape
        score = self.start_transitions[tags[:, 0]] + emissions[:, 0].gather(1, tags[:, 0:1]).squeeze(1)
        for t in range(1, T):
            step = self.transitions[tags[:, t], tags[:, t-1]] + emissions[:, t].gather(1, tags[:, t:t+1]).squeeze(1)
            score = torch.where(mask[:, t], score + step, score)
        last_idx = mask.long().sum(dim=1) - 1
        last_tags = tags.gather(1, last_idx.unsqueeze(1)).squeeze(1)
        return score + self.end_transitions[last_tags]

    def _forward_alg(self, emissions, mask):
        B, T, C = emissions.shape
        alpha = self.start_transitions.unsqueeze(0) + emissions[:, 0]
        for t in range(1, T):
            emit = emissions[:, t].unsqueeze(2)  # FIXED
            trans = self.transitions.unsqueeze(0)
            alpha_exp = alpha.unsqueeze(1) + trans + emit
            new_alpha = torch.logsumexp(alpha_exp, dim=2)
            alpha = torch.where(mask[:, t].unsqueeze(1), new_alpha, alpha)
        return torch.logsumexp(alpha + self.end_transitions.unsqueeze(0), dim=1)

    def decode(self, emissions, mask):
        B, T, C = emissions.shape
        viterbi = emissions[:, 0] + self.start_transitions.unsqueeze(0)
        bp = []
        for t in range(1, T):
            scores = viterbi.unsqueeze(1) + self.transitions.unsqueeze(0)
            best_s, best_i = scores.max(dim=2)
            new_v = best_s + emissions[:, t]
            viterbi = torch.where(mask[:, t].unsqueeze(1), new_v, viterbi)
            bp.append(best_i)
        viterbi = viterbi + self.end_transitions.unsqueeze(0)
        lengths = mask.long().sum(dim=1)
        paths = []
        for b in range(B):
            sl = lengths[b].item()
            bl = viterbi[b].argmax().item()
            path = [bl]
            for t in range(sl - 2, -1, -1):
                bl = bp[t][b, bl].item()
                path.append(bl)
            path.reverse()
            path += [0] * (T - sl)
            paths.append(path)
        return paths


class TeacherNER(nn.Module):
    def __init__(self, backbone, num_tags, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        self.num_tags = num_tags
        d = backbone.tok_emb.weight.shape[1]
        self.bilstm = nn.LSTM(d, d // 2, 2, batch_first=True, bidirectional=True, dropout=dropout)
        self.ner_dropout = nn.Dropout(dropout)
        self.ner_head = nn.Linear(d, num_tags)
        self.crf = CRF(num_tags)
        nn.init.normal_(self.ner_head.weight, std=0.02)
        nn.init.zeros_(self.ner_head.bias)

    def get_hidden(self, ids, attention_mask=None):
        B, T = ids.shape
        x = self.backbone.drop(self.backbone.tok_emb(ids))
        causal = self.backbone._causal_mask(T, ids.device)
        if attention_mask is not None:
            pm = torch.zeros(B, 1, 1, T, device=ids.device, dtype=x.dtype)
            pm.masked_fill_(attention_mask[:, None, None, :] == 0, float("-inf"))
            mask = causal + pm
        else:
            mask = causal
        for layer in self.backbone.layers:
            x, _ = layer(x, self.backbone.rope_cos, self.backbone.rope_sin, mask)
        return self.backbone.norm(x)

    def forward(self, input_ids, labels=None, attention_mask=None):
        h = self.get_hidden(input_ids, attention_mask)
        h, _ = self.bilstm(h)
        emissions = self.ner_head(self.ner_dropout(h))
        out = {"logits": emissions}
        if labels is not None:
            crf_mask = (labels != -100)
            sl = labels.clone()
            sl[~crf_mask] = 0
            out["loss"] = self.crf(emissions, sl, crf_mask)
        return out

print("Teacher model defined")

Teacher model defined


## 4 — Student Model (Encoder + CRF)

In [ ]:
# ── Encoder model (matches enc-dec checkpoint naming) ──
class RMSNorm_Enc(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.g = nn.Parameter(torch.ones(d))

    def forward(self, x):
        return self.g * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)

def build_rope_cache(seq_len, head_dim, theta=10000.0, device='cpu'):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    freqs = torch.outer(torch.arange(seq_len, device=device).float(), inv_freq)
    return freqs.cos(), freqs.sin()

def apply_rope_enc(x, cos, sin):
    T = x.size(-2)
    cos = cos[:T].unsqueeze(0).unsqueeze(0)
    sin = sin[:T].unsqueeze(0).unsqueeze(0)
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    return torch.stack([x1*cos - x2*sin, x1*sin + x2*cos], dim=-1).flatten(-2)

class GQASelfAttn_Enc(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_q = cfg.n_heads
        self.n_kv = cfg.n_kv_heads
        self.head_dim = cfg.head_dim
        self.n_rep = self.n_q // self.n_kv
        self.wq = nn.Linear(cfg.d_model, self.n_q * self.head_dim, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.n_kv * self.head_dim, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.n_kv * self.head_dim, bias=False)
        self.wo = nn.Linear(self.n_q * self.head_dim, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, attn_mask=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_q, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        q = apply_rope_enc(q, cos, sin)
        k = apply_rope_enc(k, cos, sin)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=False)
        return self.wo(out.transpose(1, 2).contiguous().view(B, T, -1))

class SwiGLU_Enc(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class EncBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1 = RMSNorm_Enc(cfg.d_model, cfg.rms_eps)
        self.attn = GQASelfAttn_Enc(cfg)
        self.n2 = RMSNorm_Enc(cfg.d_model, cfg.rms_eps)
        self.ff = SwiGLU_Enc(cfg.d_model, cfg.d_ff)

    def forward(self, x, cos, sin, amask=None):
        x = x + self.attn(self.n1(x), cos, sin, amask)
        return x + self.ff(self.n2(x))

class LegalEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.src_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.enc_blocks = nn.ModuleList([EncBlock(cfg) for _ in range(cfg.n_layers)])
        self.enc_norm = RMSNorm_Enc(cfg.d_model, cfg.rms_eps)
        cos, sin = build_rope_cache(cfg.max_seq_len * 4, cfg.head_dim, cfg.rope_theta)
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)

    def forward(self, ids, attention_mask=None):
        B, T = ids.shape
        x = self.src_emb(ids)
        amask = None
        if attention_mask is not None:
            amask = torch.zeros(B, 1, 1, T, device=ids.device, dtype=x.dtype)
            amask.masked_fill_(attention_mask[:, None, None, :] == 0, float("-inf"))
        for blk in self.enc_blocks:
            x = blk(x, self.rope_cos, self.rope_sin, amask)
        return self.enc_norm(x)


class StudentNER(nn.Module):
    def __init__(self, encoder, num_tags, dropout=0.3):
        super().__init__()
        self.encoder = encoder
        self.num_tags = num_tags
        d = encoder.cfg.d_model
        self.ner_dropout = nn.Dropout(dropout)
        self.ner_head = nn.Linear(d, num_tags)
        self.crf = CRF(num_tags)
        nn.init.normal_(self.ner_head.weight, std=0.02)
        nn.init.zeros_(self.ner_head.bias)

    def forward(self, input_ids, labels=None, attention_mask=None):
        h = self.encoder(input_ids, attention_mask)
        emissions = self.ner_head(self.ner_dropout(h))
        out = {"logits": emissions}
        if labels is not None:
            crf_mask = (labels != -100)
            sl = labels.clone()
            sl[~crf_mask] = 0
            out["loss"] = self.crf(emissions, sl, crf_mask)
        return out

print("Student model defined")

Student model defined


## 5 — Load Teacher & Student

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)

# Load teacher
print("Loading teacher...")
teacher_backbone = LegalLM(T_CFG).to(DEVICE)
teacher_model = TeacherNER(teacher_backbone, T_CFG.bio_num_tags).to(DEVICE)

t_ckpt = torch.load(TEACHER_CKPT, map_location=DEVICE, weights_only=False)
t_state = t_ckpt.get("model", t_ckpt)
teacher_model.load_state_dict(t_state, strict=True)
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False
print(f"Teacher loaded: {sum(p.numel() for p in teacher_model.parameters()):,} params (frozen)")
if "f1" in t_ckpt:
    print(f"  Teacher best F1: {t_ckpt['f1']:.4f}")

# Load student
print("Loading student encoder...")
encoder = LegalEncoder(S_CFG)
enc_ckpt = torch.load(ENCDEC_CKPT, map_location="cpu", weights_only=False)
enc_state = enc_ckpt.get("model", enc_ckpt)
enc_keys = {k: v for k, v in enc_state.items()
            if k.startswith(("src_emb", "enc_blocks", "enc_norm"))}
encoder.load_state_dict(enc_keys, strict=False)

student_model = StudentNER(encoder, S_CFG.bio_num_tags).to(DEVICE)
for p in student_model.parameters():
    p.requires_grad = True
print(f"Student loaded: {sum(p.numel() for p in student_model.parameters()):,} params (trainable)")

torch.cuda.empty_cache()
print("Both models loaded!")

Loading teacher...
Teacher loaded: 157,397,692 params (frozen)
Loading student encoder...
Student loaded: 48,356,540 params (trainable)
Both models loaded!


## 6 — Build Dataset

In [ ]:
all_text={**fold1_text,**fold2_text}; all_gold={**fold1_gold,**fold2_gold}
doc_ids=sorted(all_text.keys()); random.seed(CFG.seed); random.shuffle(doc_ids)
split_idx=len(doc_ids)-3
train_text={d:all_text[d] for d in doc_ids[:split_idx]}; train_gold={d:all_gold[d] for d in doc_ids[:split_idx]}
val_text={d:all_text[d] for d in doc_ids[split_idx:]}; val_gold={d:all_gold[d] for d in doc_ids[split_idx:]}
print(f"Train: {len(train_text)}, Val: {len(val_text)}")

train_ds=NERDataset(train_text,train_gold,tokenizer,CFG,tag2id)
val_ds=NERDataset(val_text,val_gold,tokenizer,CFG,tag2id)

print(f"No oversampling — {len(train_ds.windows)} windows")

train_dl=DataLoader(train_ds,batch_size=CFG.batch_size,shuffle=True,num_workers=4,pin_memory=True,drop_last=True)
val_dl=DataLoader(val_ds,batch_size=CFG.batch_size,shuffle=False,num_workers=4,pin_memory=True)

Train: 67, Val: 3


Building dataset:   0%|          | 0/67 [00:00<?, ?it/s]

  1531 windows


Building dataset:   0%|          | 0/3 [00:00<?, ?it/s]

  61 windows
No oversampling — 1531 windows


## 7 — KD Training Loop

In [ ]:
def get_cosine_mult(step, total_steps, warmup_steps):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * prog))


TRIM_CHARS = set('()[]{},:;/\\\'\"-!?.')

def _trim_spans(spans, text):
    out = []
    for s, e, l in spans:
        if s >= e:
            continue
        snip = text[s:e]
        i = 0
        while i < len(snip) and snip[i] in TRIM_CHARS:
            i += 1
        j = len(snip)
        while j > i and snip[j-1] in TRIM_CHARS:
            j -= 1
        if j > i:
            out.append([s + i, s + j, l])
    return out


def tag_preds_to_char_spans(tag_preds, char_spans, text):
    spans = []
    cur = None
    ss = None
    se = None
    for i, tid in enumerate(tag_preds):
        tn = id2tag.get(tid, "O")
        cs, ce = char_spans[i]
        if tn.startswith("B-"):
            if cur and ss is not None:
                spans.append([ss, se, cur])
            cur = tn[2:]
            ss = cs
            se = ce
        elif tn.startswith("I-"):
            if cur == tn[2:]:
                se = max(se, ce) if se else ce
            else:
                if cur and ss is not None:
                    spans.append([ss, se, cur])
                cur = tn[2:]
                ss = cs
                se = ce
        else:
            if cur and ss is not None:
                spans.append([ss, se, cur])
            cur = None
            ss = None
            se = None
    if cur and ss is not None:
        spans.append([ss, se, cur])
    return _trim_spans(spans, text)


@torch.no_grad()
def evaluate_ner_doc_level(model, text_dict, gold_dict, tokenizer, cfg, device=DEVICE):
    model.eval()
    pred_all = defaultdict(set)
    gold_all = defaultdict(set)
    tot = 0.0
    nb = 0

    for doc_id in text_dict:
        text = get_doc_text(text_dict[doc_id])
        if not text.strip():
            continue
        aids, acs = encode_with_offsets(tokenizer, text)
        n = len(aids)
        if n == 0:
            continue

        gold_anns = gold_dict.get(doc_id, [])
        full_tags = assign_bio_tags(acs, gold_anns, tag2id)

        step = cfg.window_stride if cfg.window_stride > 0 else cfg.window_size
        starts = list(range(0, max(1, n - cfg.window_size + 1), step))
        if starts[-1] + cfg.window_size < n:
            starts.append(max(0, n - cfg.window_size))

        tl = [None] * n
        td = [-1] * n

        for s in starts:
            e = min(s + cfg.window_size, n)
            wi = aids[s:e]
            wt = full_tags[s:e]
            pl = cfg.window_size - len(wi)
            inp = torch.tensor([wi + [0]*pl], dtype=torch.long, device=device)
            lb = torch.tensor([wt + [-100]*pl], dtype=torch.long, device=device)
            am = torch.tensor([[1]*len(wi) + [0]*pl], dtype=torch.long, device=device)

            with torch.amp.autocast('cuda', enabled=cfg.use_amp):
                out = model(inp, labels=lb, attention_mask=am)

            tot += out["loss"].item()
            nb += 1
            lo = out["logits"][0, :len(wi), :]

            for iw in range(len(wi)):
                g = s + iw
                d = min(iw, len(wi) - 1 - iw)
                if d > td[g]:
                    td[g] = d
                    tl[g] = lo[iw]

        el = [tl[i] if tl[i] is not None else torch.zeros(model.num_tags, device=device)
              for i in range(n)]
        em = torch.stack(el).unsqueeze(0)
        cm = torch.ones(1, n, dtype=torch.bool, device=device)
        tp = model.crf.decode(em, cm)[0]

        pc = tag_preds_to_char_spans(tp, acs, text)
        for ps, pe, pl in pc:
            pred_all[pl].add((doc_id, ps, pe))
        for ann in gold_anns:
            gs, ge, l = int(ann[0]), int(ann[1]), ann[2]
            if l in LABELS:
                gold_all[l].add((doc_id, gs, ge))

    pl = {}
    for l in LABELS:
        ps = pred_all.get(l, set())
        gs = gold_all.get(l, set())
        tp = len(ps & gs)
        fp = len(ps - gs)
        fn = len(gs - ps)
        p = tp / (tp + fp) if tp + fp > 0 else 0
        r = tp / (tp + fn) if tp + fn > 0 else 0
        pl[l] = 2 * p * r / (p + r) if p + r > 0 else 0

    return tot / max(nb, 1), sum(pl.values()) / len(LABELS), pl


def train_kd(teacher, student, train_dl, val_text, val_gold, cfg, exp_name,
             device=DEVICE, patience=7):
    set_seed(cfg.seed)
    student = student.to(device)
    teacher = teacher.to(device)
    teacher.eval()

    enc_params = [(n, p) for n, p in student.named_parameters()
                  if "encoder" in n and p.requires_grad]
    head_params = [(n, p) for n, p in student.named_parameters()
                   if "encoder" not in n and p.requires_grad]
    head_lr = getattr(cfg, 'head_lr', 5e-4)

    pg = []
    if enc_params:
        pg.append({"params": [p for _, p in enc_params], "lr": cfg.lr})
    if head_params:
        pg.append({"params": [p for _, p in head_params], "lr": head_lr})
    print(f"Encoder: {sum(p.numel() for _, p in enc_params):,} | "
          f"Head: {sum(p.numel() for _, p in head_params):,}")

    opt = torch.optim.AdamW(pg, weight_decay=cfg.weight_decay)
    base_lrs = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler('cuda', enabled=cfg.use_amp)
    total_steps = len(train_dl) * cfg.num_epochs // cfg.grad_accum

    best_f1 = 0.0
    best_state = None
    gs = 0
    no_imp = 0
    all_t = [p for p in student.parameters() if p.requires_grad]
    alpha = cfg.kd_alpha
    T = cfg.kd_temperature

    for epoch in range(cfg.num_epochs):
        student.train()
        rl = 0.0
        t0 = time.time()

        for bi, batch in enumerate(train_dl):
            ids = batch["input_ids"].to(device)
            labs = batch["labels"].to(device)
            am = batch["attention_mask"].to(device)

            with torch.amp.autocast('cuda', enabled=cfg.use_amp):
                # Student forward
                s_out = student(ids, labels=labs, attention_mask=am)
                hard_loss = s_out["loss"]

                # Teacher forward (no grad)
                with torch.no_grad():
                    t_out = teacher(ids, attention_mask=am)
                    t_logits = t_out["logits"]

                # KD loss: KL div on softened emissions
                valid = (labs != -100)
                if valid.any():
                    s_log = F.log_softmax(s_out["logits"][valid] / T, dim=-1)
                    t_prob = F.softmax(t_logits[valid] / T, dim=-1)
                    kd_loss = F.kl_div(s_log, t_prob, reduction='batchmean') * (T * T)
                else:
                    kd_loss = torch.tensor(0.0, device=device)

                loss = (alpha * hard_loss + (1 - alpha) * kd_loss) / cfg.grad_accum

            scaler.scale(loss).backward()
            rl += loss.item()

            if (bi + 1) % cfg.grad_accum == 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(all_t, cfg.max_grad_norm)
                mult = get_cosine_mult(gs, total_steps, cfg.warmup_steps)
                for g, blr in zip(opt.param_groups, base_lrs):
                    g["lr"] = max(cfg.min_lr, blr * mult)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                gs += 1

            if (bi + 1) % (50 * cfg.grad_accum) == 0:
                print(f"  [{exp_name}] Ep {epoch+1} step {bi+1}/{len(train_dl)} "
                      f"| loss {rl/(bi+1):.4f} | {time.time()-t0:.0f}s")

        vl, mf1, pl = evaluate_ner_doc_level(student, val_text, val_gold,
                                              tokenizer, cfg, device)
        print(f"  [{exp_name}] Epoch {epoch+1} | val_loss={vl:.4f} | DOC macro_f1={mf1:.4f}")
        for l, f1 in sorted(pl.items(), key=lambda x: -x[1]):
            print(f"    {l:30s} F1={f1:.4f}")

        if mf1 > best_f1:
            best_f1 = mf1
            best_state = copy.deepcopy(student.state_dict())
            torch.save({"model": best_state, "f1": best_f1, "epoch": epoch + 1},
                       CKPT_DIR / f"{exp_name}_best.pt")
            print(f"  New best F1={best_f1:.4f}")
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    if best_state:
        student.load_state_dict(best_state)
    print(f"  Done. Best F1: {best_f1:.4f}")
    return student

print("KD training defined")

KD training defined


## 8 — Run KD

In [ ]:
cfg_kd = copy.copy(CFG)
cfg_kd.lr = 1e-5
cfg_kd.head_lr = 5e-4
cfg_kd.warmup_steps = 200
cfg_kd.num_epochs = 25
cfg_kd.kd_alpha = 0.5        # 50% hard CRF loss + 50% soft KD loss
cfg_kd.kd_temperature = 2.0

student_model = train_kd(teacher_model, student_model, train_dl,
                         val_text, val_gold, cfg_kd, "exp_kd_bio", patience=7)


# Predict
@torch.no_grad()
def predict_test(model, text_dict, tokenizer, cfg, device=DEVICE):
    model.eval()
    predictions = {}
    for doc_id in tqdm(text_dict, desc="Predicting"):
        text = get_doc_text(text_dict[doc_id])
        if not text.strip():
            predictions[doc_id] = []
            continue
        aids, acs = encode_with_offsets(tokenizer, text)
        n = len(aids)
        if n == 0:
            predictions[doc_id] = []
            continue

        step = cfg.window_stride if cfg.window_stride > 0 else cfg.window_size
        starts = list(range(0, max(1, n - cfg.window_size + 1), step))
        if starts[-1] + cfg.window_size < n:
            starts.append(max(0, n - cfg.window_size))

        tl = [None] * n
        td = [-1] * n

        for s in starts:
            e = min(s + cfg.window_size, n)
            wi = aids[s:e]
            pl = cfg.window_size - len(wi)
            inp = torch.tensor([wi + [0]*pl], dtype=torch.long, device=device)
            am = torch.tensor([[1]*len(wi) + [0]*pl], dtype=torch.long, device=device)

            with torch.amp.autocast('cuda', enabled=cfg.use_amp):
                out = model(inp, attention_mask=am)

            lo = out["logits"][0, :len(wi), :]
            for iw in range(len(wi)):
                g = s + iw
                d = min(iw, len(wi) - 1 - iw)
                if d > td[g]:
                    td[g] = d
                    tl[g] = lo[iw]

        el = [tl[i] if tl[i] is not None else torch.zeros(model.num_tags, device=device)
              for i in range(n)]
        em = torch.stack(el).unsqueeze(0)
        cm = torch.ones(1, n, dtype=torch.bool, device=device)
        predictions[doc_id] = tag_preds_to_char_spans(
            model.crf.decode(em, cm)[0], acs, text)
    return predictions


preds = predict_test(student_model, fold3_text, tokenizer, CFG)

pred_path = SUB_DIR / "prediction_exp_kd_bio.json"
zip_path = SUB_DIR / "submission_exp_kd_bio.zip"
with open(pred_path, "w") as f:
    json.dump(preds, f, ensure_ascii=False)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, "prediction.json")
print(f"Submission: {len(preds)} docs, {sum(len(v) for v in preds.values())} entities -> {zip_path.name}")

del teacher_model
torch.cuda.empty_cache()
print("Done!")

Encoder: 48,339,840 | Head: 16,700
  [exp_kd_bio] Ep 1 step 200/382 | loss 210.2231 | 115s
  [exp_kd_bio] Epoch 1 | val_loss=629.8117 | DOC macro_f1=0.0005
    AUTH                           F1=0.0060
    APP                            F1=0.0000
    RESP                           F1=0.0000
    A.COUNSEL                      F1=0.0000
    R.COUNSEL                      F1=0.0000
    JUDGE                          F1=0.0000
    WIT                            F1=0.0000
    COURT                          F1=0.0000
    STAT                           F1=0.0000
    PREC                           F1=0.0000
    DATE                           F1=0.0000
    CASENO                         F1=0.0000
  New best F1=0.0005
  [exp_kd_bio] Ep 2 step 200/382 | loss 41.3709 | 113s
  [exp_kd_bio] Epoch 2 | val_loss=126.6572 | DOC macro_f1=0.0461
    DATE                           F1=0.4318
    COURT                          F1=0.1026
    CASENO                         F1=0.0190
    APP                     

In [ ]:
# Save results to Drive
drive_out = Path("/content/drive/MyDrive/phase2_ner_results")
drive_out.mkdir(parents=True, exist_ok=True)

for zf in sorted(SUB_DIR.glob("submission_*.zip")):
    shutil.copy(zf, drive_out / zf.name)
    print(f"Saved to Drive: {drive_out / zf.name}")

best_ckpt = CKPT_DIR / "exp_kd_bio_best.pt"
if best_ckpt.exists():
    shutil.copy(best_ckpt, drive_out / best_ckpt.name)
    print(f"Saved checkpoint to Drive: {drive_out / best_ckpt.name}")

from google.colab import files
for zf in sorted(SUB_DIR.glob("submission_*.zip")):
    files.download(str(zf))
print("Done!")